<a href="https://colab.research.google.com/github/pablonvsx/pisi3-ufrpe/blob/main/data-science/notebooks/ML/experimentos_amostra_bin%C3%A1ria/CONSTRU%C3%87%C3%83O_DE_CLASSES_BIN%C3%81RIAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORT DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/Cópia de water_quality_2000_2008_novorotulo.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/water_quality_2000_2008.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (59896, 23)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status,Year
0,Canada,FISW_32,Lake,2003-12-02,0.043792,2.13333,9.824,0.00200,7.7900,12.00000,...,Excellent,1,1,1,1,2.0,1,5,Adequada,2003
1,Canada,IEEA_10_32,Lake,2001-06-08,0.015920,0.55000,9.824,0.00400,7.7900,16.80000,...,Excellent,1,1,1,1,2.0,1,5,Adequada,2001
2,Canada,CHRW-1876,River,2000-01-12,0.064400,10.87500,11.250,0.03590,8.2833,12.76150,...,Good,1,1,0,1,1.0,1,4,Boa,2000
3,Canada,ES063ESPFAA0000714,River,2004-01-12,1.071725,1.24444,5.850,0.20425,7.1000,18.32500,...,Fair,1,1,1,0,3.7,1,4,Boa,2004
4,Canada,CZPLA_391,River,2003-01-12,0.039740,1.83333,11.050,0.06100,7.7500,8.66667,...,Good,1,1,1,0,2.0,1,4,Boa,2003


In [ ]:
df.columns

Index(['Country', 'Area', 'Waterbody Type', 'Date', 'Ammonia (mg/l)',
       'Biochemical Oxygen Demand (mg/l)', 'Dissolved Oxygen (mg/l)',
       'Orthophosphate (mg/l)', 'pH (ph units)', 'Temperature (cel)',
       'Nitrogen (mg/l)', 'Nitrate (mg/l)', 'CCME_Values', 'CCME_WQI', 'ph_ok',
       'od_ok', 'dbo_ok', 'nitrate_ok', 'ammonia_limit', 'ammonia_ok',
       'environmental_score', 'conama_status', 'Year'],
      dtype='object')

In [ ]:
def classificar_binario(score):
    if score == 5:
        return "Adequada"
    else:
        return "Não adequada"

In [ ]:
df["conama_status"] = df["environmental_score"].apply(classificar_binario)

In [ ]:
df["conama_status"].value_counts()

,count
conama_status,
Adequada,41169
Não adequada,18727


In [ ]:
round(df["conama_status"].value_counts(normalize=True) * 100, 2)

,proportion
conama_status,
Adequada,68.73
Não adequada,31.27


In [ ]:
pd.crosstab(
    df["environmental_score"],
    df["conama_status"]
)

conama_status,Adequada,Não adequada
environmental_score,,
1,0,11
2,0,1078
3,0,5842
4,0,11796
5,41169,0


In [ ]:
df.to_parquet(
    "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada_binaria_2000_2008.parquet",
    index=False
)

In [ ]:
df.shape

(59896, 24)

In [ ]:
df.head()

,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status,Year,classificacao_binaria
0,Canada,FISW_32,Lake,2003-12-02,0.043792,2.13333,9.824,0.00200,7.7900,12.00000,...,1,1,1,1,2.0,1,5,Adequada,2003,Adequada
1,Canada,IEEA_10_32,Lake,2001-06-08,0.015920,0.55000,9.824,0.00400,7.7900,16.80000,...,1,1,1,1,2.0,1,5,Adequada,2001,Adequada
2,Canada,CHRW-1876,River,2000-01-12,0.064400,10.87500,11.250,0.03590,8.2833,12.76150,...,1,1,0,1,1.0,1,4,Não adequada,2000,Não adequada
3,Canada,ES063ESPFAA0000714,River,2004-01-12,1.071725,1.24444,5.850,0.20425,7.1000,18.32500,...,1,1,1,0,3.7,1,4,Não adequada,2004,Não adequada
4,Canada,CZPLA_391,River,2003-01-12,0.039740,1.83333,11.050,0.06100,7.7500,8.66667,...,1,1,1,0,2.0,1,4,Não adequada,2003,Não adequada


In [ ]:
df.drop(columns=["classificacao_binaria"], inplace=True)

In [ ]:
df.columns

Index(['Country', 'Area', 'Waterbody Type', 'Date', 'Ammonia (mg/l)',
       'Biochemical Oxygen Demand (mg/l)', 'Dissolved Oxygen (mg/l)',
       'Orthophosphate (mg/l)', 'pH (ph units)', 'Temperature (cel)',
       'Nitrogen (mg/l)', 'Nitrate (mg/l)', 'CCME_Values', 'CCME_WQI', 'ph_ok',
       'od_ok', 'dbo_ok', 'nitrate_ok', 'ammonia_limit', 'ammonia_ok',
       'environmental_score', 'conama_status', 'Year'],
      dtype='object')

In [ ]:
df.head()

,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status,Year
0,Canada,FISW_32,Lake,2003-12-02,0.043792,2.13333,9.824,0.00200,7.7900,12.00000,...,Excellent,1,1,1,1,2.0,1,5,Adequada,2003
1,Canada,IEEA_10_32,Lake,2001-06-08,0.015920,0.55000,9.824,0.00400,7.7900,16.80000,...,Excellent,1,1,1,1,2.0,1,5,Adequada,2001
2,Canada,CHRW-1876,River,2000-01-12,0.064400,10.87500,11.250,0.03590,8.2833,12.76150,...,Good,1,1,0,1,1.0,1,4,Não adequada,2000
3,Canada,ES063ESPFAA0000714,River,2004-01-12,1.071725,1.24444,5.850,0.20425,7.1000,18.32500,...,Fair,1,1,1,0,3.7,1,4,Não adequada,2004
4,Canada,CZPLA_391,River,2003-01-12,0.039740,1.83333,11.050,0.06100,7.7500,8.66667,...,Good,1,1,1,0,2.0,1,4,Não adequada,2003


In [ ]:
df.to_parquet(
    "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada_binaria_2000_2008.parquet",
    index=False
)